# M0 — train the arXiv category classifier

Fine-tunes `distilbert-base-uncased` on arXiv primary categories, writes the
artifact + `model_card.json`, and pushes to the HuggingFace Hub.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run the cells top to bottom. Total ~15–25 min.

## 1. Confirm the GPU

Fails loudly here rather than 20 minutes into a CPU run.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Dependencies

In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn

## 3. Clone the repo

Always from `/content`, always fresh — a second clone inside an existing one
gives you `arxiv-classifier-api/arxiv-classifier-api/` and a very confusing
afternoon. Discards uncommitted edits made inside Colab.

In [ ]:
%cd /content
!rm -rf arxiv-classifier-api
!git clone -q https://github.com/eren-o23/arxiv-classifier-api.git
%cd /content/arxiv-classifier-api
!git log --oneline -1

## 4. HuggingFace token

Add a secret named `HF_TOKEN` (🔑 in the left sidebar, **Notebook access** on)
with a **write** token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

Colab secrets only resolve inside this kernel, so it has to go into the
environment for the `!python` subprocess below to see it.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 5. Look at the data first

`categories` is a list like `["Computer Science Archive->cs.CV", ...]`. The first
entry is arXiv's primary category; `train.py` takes its leaf.

In [ ]:
!python training/train.py --inspect

## 6. Train

~13 min on a T4 in fp16. If the first line reports that `fp16` was dropped,
it is running fp32 at roughly half speed — slower, not broken.

Only ~14% of the dataset has a primary category in our ten labels, so expect
about **22k / 2.8k / 2.8k** rows, not the 32k/4k/4k the spec assumed.

In [ ]:
!python training/train.py

## 7. Check the result before believing it

- top-1 **0.75–0.82**, top-3 **above 0.90**. Below ~0.72, try 3 epochs.
- **macro-F1 far below top-1** means the rare classes (cs.NI, cs.DS) are being
  ignored rather than learned — arXiv is lopsided toward cs.LG and cs.CV.
- Errors should **cluster in the cs.LG / cs.AI / stat.ML block**. Spread evenly
  across all ten classes means the labels got shuffled in the map step, and the
  accuracy number would still look plausible.
- The test split is only ~2.8k, so the accuracy carries roughly ±1.5 points of
  sampling noise. Don't tune on a one-point difference.

**Copy the Hub commit SHA the cell above printed.** M4's Dockerfile pins to it.

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("docs/model_eval.md").read()))
display(Image("docs/confusion_matrix.png"))

## 8. Retrieve the evidence

`models/` is gitignored, so `docs/` is the only place these survive. Download
both and commit them from your machine.

In [ ]:
from google.colab import files

files.download("docs/model_eval.md")
files.download("docs/confusion_matrix.png")

---

**M0 is not done yet.** Verify locally, from the repo root:

```bash
hf download <your-hf-username>/arxiv-classifier-v1 --local-dir models/arxiv-v1
uv run --python 3.11 --with torch --with transformers \
  scripts/check_model.py --title "..." --abstract "..."
```

Paste a real abstract from arXiv in an obvious category. Correct label → tick
M0 in the README.